In [1]:
import sys
#lets you access python's runtime environment
from pathlib import Path
#sys.path is a built in variable in the sys module and contains a list of directories that is seached through when you do an import
#so we are appending the src directory to that
sys.path.append(str(Path().resolve().parent / "src"))
import config
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance


panel_data_path = config.PROJECT_ROOT/ "data" / "panel_data.csv"
panel_data = pd.read_csv(panel_data_path)

In [2]:
#### TODO move this to data transformation
panel_data['Churns'] = ~panel_data['CompetesNextYear']
panel_data = panel_data.drop(columns = 'CompetesNextYear')

panel_data['Date'] = pd.to_datetime(panel_data['Date'])
panel_data = panel_data.sort_values('Date', ascending= True)



###TODO 
#add this to data_transformation
panel_data['CovidAffectedPrediction'] = (
    panel_data['Year'].isin([2019])
).astype(int)
panel_data = panel_data.drop(columns = 'Date')

In [3]:
#move to data_transformation.ipynb
###TODO
panel_data = panel_data.drop(columns = ['FederationRankCapped', 'FederationCount'])

### Train/validate/test split
Train/validate/test split must be time-aware. Entries in the train set should predate entries in the validation set which should predate entries in the test set. The dataframe year_counts is used to decide which years should be used for the train/validation/test sets.

In [4]:
year_counts = panel_data.groupby('Year').size().reset_index(name = 'NumberOfRows')
year_counts = year_counts.sort_values('Year')
year_counts['Cumulative'] = year_counts['NumberOfRows'].cumsum()
total_rows = year_counts.loc[len(year_counts)-1, 'Cumulative']
year_counts['EntriesPrecedingAndIncluding(%)'] = (year_counts['Cumulative']/total_rows) *100
year_counts

,Year,NumberOfRows,Cumulative,EntriesPrecedingAndIncluding(%)
0,2015,20521,20521,6.723501
1,2016,26591,47112,15.435778
2,2017,31939,79051,25.900273
3,2018,37641,116692,38.232972
4,2019,40916,157608,51.638692
5,2020,17673,175281,57.429074
6,2021,25873,201154,65.906105
7,2022,27153,228307,74.802515
8,2023,35947,264254,86.580192
9,2024,40959,305213,100.000000


- 2022 and before used as train set
- 2023 used for validation set
- 2024 used for test set

In [5]:
#drop WeightClassKg in data_transformation instead in refactoring
### TODO
panel = panel_data.drop(columns = ['Name', 'WeightClassKg'])
train = panel[panel['Year']<=2022].copy()
validate = panel[panel['Year']==2023].copy()
test = panel[panel['Year']>2023].copy()

train_X = train.drop(columns = 'Churns')
train_y = train['Churns']
validate_X = validate.drop(columns = 'Churns')
validate_y = validate['Churns']
test_X = test.drop(columns = 'Churns')
test_y = test['Churns']

In [6]:
panel_data[['PercentageImprovementGradientWithinYear', 'ImprovementGradientWithinYear']].corr()

,PercentageImprovementGradientWithinYear,ImprovementGradientWithinYear
PercentageImprovementGradientWithinYear,1.000000,0.970931
ImprovementGradientWithinYear,0.970931,1.000000


In [7]:
panel_data[['PercentageImprovementGradientTwoMeets', 'ImprovementGradientTwoMeets']].corr()

,PercentageImprovementGradientTwoMeets,ImprovementGradientTwoMeets
PercentageImprovementGradientTwoMeets,1.000000,0.938697
ImprovementGradientTwoMeets,0.938697,1.000000


In [8]:
panel_data[['PercentageImprovementGradientBetweenYears', 'ImprovementGradientBetweenYears']].corr()

,PercentageImprovementGradientBetweenYears,ImprovementGradientBetweenYears
PercentageImprovementGradientBetweenYears,1.000000,0.865184
ImprovementGradientBetweenYears,0.865184,1.000000


In [9]:
panel_data[['ImprovementAccelerationForPercentage', 'ImprovementAcceleration']].corr()

,ImprovementAccelerationForPercentage,ImprovementAcceleration
ImprovementAccelerationForPercentage,1.000000,0.957421
ImprovementAcceleration,0.957421,1.000000


In [10]:
#will be using performance on validation set for feature selection
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
def evaluate_model(classifier, X_val, y_val):
    preds = classifier.predict(X_val)
    probs = classifier.predict_proba(X_val)[:, 1]
    return {
        "f1":        f1_score(y_val, preds),
        "recall":    recall_score(y_val, preds),
        "accuracy":  accuracy_score(y_val, preds),
        "precision": precision_score(y_val, preds),
        "roc_auc":   roc_auc_score(y_val, probs)
        
    }

In [11]:
clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42,
    categorical_features = ['Sex']
)
clf.fit(train_X, train_y)



#performance of classifier in validation set when trained on all features. will be used for comparison.
all_feats_metrics = evaluate_model(clf, validate_X, validate_y)

train_perc_X = train_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])

validate_perc_X = validate_X.drop(columns = ['ImprovementGradientWithinYear', 'ImprovementGradientTwoMeets', 
                                           'ImprovementGradientBetweenYears', 'ImprovementAcceleration'])


clf_perc = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42,
    categorical_features = ['Sex']
)
clf_perc.fit(train_perc_X, train_y)

perc_metrics = evaluate_model(clf_perc, validate_perc_X, validate_y)


train_raw_X = train_X.drop(columns = ['PercentageImprovementGradientWithinYear', 'PercentageImprovementGradientTwoMeets', 
                                           'PercentageImprovementGradientBetweenYears', 'ImprovementAccelerationForPercentage'])

validate_raw_X = validate_X.drop(columns = ['PercentageImprovementGradientWithinYear', 'PercentageImprovementGradientTwoMeets', 
                                           'PercentageImprovementGradientBetweenYears', 'ImprovementAccelerationForPercentage'])


clf_raw = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42
)
clf_raw.fit(train_raw_X, train_y)

raw_metrics = evaluate_model(clf_raw, validate_raw_X, validate_y)

print(all_feats_metrics)
print(perc_metrics)
print(raw_metrics)

{'f1': 0.6727949916792139, 'recall': 0.6852668962548428, 'accuracy': 0.65540935265808, 'precision': 0.6607689513827635, 'roc_auc': 0.7078558178311439}
{'f1': 0.6716072993864053, 'recall': 0.680262591476539, 'accuracy': 0.6560770022533173, 'precision': 0.6631694906363111, 'roc_auc': 0.710456202494312}
{'f1': 0.6731689265984723, 'recall': 0.685213086526044, 'accuracy': 0.6560213647870476, 'precision': 0.6615408592654164, 'roc_auc': 0.7080472525346884}


### Establish baseline accuracy

In [12]:
clf_baseline = DummyClassifier(strategy="most_frequent")
clf_baseline.fit(train_X, train_y)
baseline_accuracy = clf_baseline.score(test_X, test_y)
evaluate_model(clf_baseline, validate_X, validate_y)

{'f1': 0.6815939557316022,
 'recall': 1.0,
 'accuracy': 0.5169833365788522,
 'precision': 0.5169833365788522,
 'roc_auc': 0.5}

permutation importance and then drop features with negative importance. 


In [13]:
result = permutation_importance(clf_raw, validate_raw_X, validate_y, n_repeats=10, random_state=42)

feature_importance_perm = pd.DataFrame({
    'Feature': validate_raw_X.columns,
    'Importance': result.importances_mean
}).sort_values('Importance', ascending=False, ignore_index=True)

perm_feats = feature_importance_perm.loc[feature_importance_perm['Importance']>0, 'Feature'].to_list()
train_perm_X = train_X[perm_feats]
validate_perm_X = validate_X[perm_feats]
clf_perm_feats = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42,
    categorical_features = ['Sex']
)
clf_perm_feats.fit(train_perm_X, train_y)

perm_metrics = evaluate_model(clf_perm_feats, validate_perm_X, validate_y)

In [14]:
feature_importance_perm

,Feature,Importance
0,TimeSinceLastPBYearEnd,3.334632e-02
1,NumberOfMeets,2.261663e-02
2,Age,1.619328e-02
3,Goodlift,1.440454e-02
4,AvgMeetsPerYear,7.694662e-03
5,ImprovementGradientBetweenYears,2.014076e-03
6,Sex,1.351990e-03
7,ImprovementAcceleration,7.232871e-04
8,ImprovementGradientTwoMeets,3.894623e-04
9,TimeCompetingYearEnd,3.532979e-04


In [15]:
perm_metrics

{'f1': 0.697206758712019,
 'recall': 0.7649052948773138,
 'accuracy': 0.6565221019834757,
 'precision': 0.6405172802234939,
 'roc_auc': 0.7078323545208441}

### How many features to keep

Will take top N features where N is determined by taking top n features and seeing which value of n yields best performance on validation set. Can then use top N features to retrain classifier and evaluate performance using test set.


In [16]:
ablation_feats = perm_feats
ablation_feats

['TimeSinceLastPBYearEnd',
 'NumberOfMeets',
 'Age',
 'Goodlift',
 'AvgMeetsPerYear',
 'ImprovementGradientBetweenYears',
 'Sex',
 'ImprovementAcceleration',
 'ImprovementGradientTwoMeets',
 'TimeCompetingYearEnd',
 'LastMeetAttemptsMade',
 'FederationProportion',
 'PercentageBombOuts',
 'ImprovementGradientWithinYear']

In [17]:
# ablation_df = pd.DataFrame({
#     'Feature': train_perc_X.columns,
#     'Importance': importances_ablation
# }).sort_values('Importance', ascending=False, ignore_index =True)
# ablation_df

# reduced_feats = feature_importance_df.loc[feature_importance_df['Importance']>0.01]
# reduced_feats = reduced_feats.drop(labels = [2,4], axis = 0).reset_index(drop = True)

In [ ]:
results = []

for i in tqdm(range(1,len(ablation_feats))):
    features = ablation_feats[:i]

    train_n_X = train_perm_X[features]
    validate_n_X = validate_perm_X[features]

    categorical_features = ['Sex'] if 'Sex' in features else None
    clf_n = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=200,
    max_depth=6,
    random_state=42,
    categorical_features = categorical_features
)
    clf_n.fit(train_n_X, train_y)
    
    preds_n = clf_n.predict(validate_n_X)
    probs_n = clf_n.predict_proba(validate_n_X)[:, 1]
    acc_n = accuracy_score(validate_y, preds_n)
    f1_n = f1_score(validate_y, preds_n)
    precision_n = precision_score(validate_y, preds_n)
    recall_n = recall_score(validate_y, preds_n)
    roc_auc_n = roc_auc_score(validate_y, probs_n)

    results.append({'features': features, 
                    'accuracy': acc_n,'f1': f1_n, 'precision': precision_n, 
                    'recall':recall_n, 'roc_auc': roc_auc_n})

results_df = pd.DataFrame(results)

 85%|███████████████████████████████████████████████████████████████████████████████████████████████████▊                  | 11/13 [00:48<00:09,  4.70s/it]

In [ ]:
results_df

In [ ]:

results_df['feature_added'] = results_df['features'].apply(lambda x: x[-1])
results_df

In [ ]:
n_features = range(1, len(results_df) + 1)

plt.figure(figsize=(10, 5))
plt.plot(n_features, results_df['f1'], marker='o')
plt.xticks(n_features)
plt.xlabel('Number of Features')
plt.ylabel('F1')
plt.title('Ablation: f1')
plt.grid(True)
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(n_features, results_df['recall'], marker='o')
plt.xticks(n_features)
plt.xlabel('Number of Features')
plt.ylabel('Recall')
plt.title('Ablation: recall')
plt.grid(True)
plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 5))
plt.plot(n_features, results_df['precision'], marker='o')
plt.xticks(n_features)
plt.xlabel('Number of Features')
plt.ylabel('Recall')
plt.title('Ablation: precision')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
print(f"Max F1 at {results_df['f1'].idxmax() + 1} features: {results_df['f1'].max():.4f}")
print(f"Max Recall at {results_df['recall'].idxmax() + 1} features: {results_df['recall'].max():.4f}")
print(f"Max Precision at {results_df['precision'].idxmax() + 1} features: {results_df['precision'].max():.4f}")
print(f"Max Accuracy at {results_df['accuracy'].idxmax() + 1} features: {results_df['accuracy'].max():.4f}")

9 features
Prioritising recall slightly more than precision and 9 features has high f1

In [ ]:
n_features = range(1, len(results_df) + 1)

fig, (ax1, ax2, ax3, ax4) = plt.subplots(4,1, figsize=(10, 10))

ax1.plot(n_features, results_df['f1'], marker='o')
ax1.set_xticks(n_features)
ax1.set_xlabel('Number of Features')
ax1.set_ylabel('F1')
ax1.set_title('Ablation: F1')
ax1.grid(True)

ax2.plot(n_features, results_df['recall'], marker='o')
ax2.set_xticks(n_features)
ax2.set_xlabel('Number of Features')
ax2.set_ylabel('Recall')
ax2.set_title('Ablation: Recall')
ax2.grid(True)

ax3.plot(n_features, results_df['precision'], marker='o')
ax3.set_xticks(n_features)
ax3.set_xlabel('Number of Features')
ax3.set_ylabel('Precision')
ax3.set_title('Ablation: Precision')
ax3.grid(True)


ax4.plot(n_features, results_df['accuracy'], marker='o')
ax4.set_xticks(n_features)
ax4.set_xlabel('Number of Features')
ax4.set_ylabel('Accuracy')
ax4.set_title('Ablation: Accuracy')
ax4.grid(True)

plt.tight_layout()
plt.show()